# Module 5: Attention & Transformers

This notebook implements the attention mechanism and Transformer architecture.

**Topics covered:**
- Scaled dot-product attention
- Multi-head attention
- Positional encoding
- Transformer blocks

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
%matplotlib inline

## 5.1 Scaled Dot-Product Attention

In [ ]:
def softmax(x, axis=-1):
    """Numerically stable softmax."""
    x = x - np.max(x, axis=axis, keepdims=True)
    exp_x = np.exp(x)
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)

def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Compute scaled dot-product attention.
    
    Attention(Q, K, V) = softmax(QK^T / sqrt(d_k)) V
    
    Args:
        Q: Queries (..., seq_len_q, d_k)
        K: Keys (..., seq_len_k, d_k)
        V: Values (..., seq_len_k, d_v)
        mask: Optional mask (..., seq_len_q, seq_len_k)
    
    Returns:
        output: Attention output (..., seq_len_q, d_v)
        weights: Attention weights (..., seq_len_q, seq_len_k)
    """
    d_k = Q.shape[-1]
    
    # Compute attention scores
    scores = Q @ K.swapaxes(-2, -1) / np.sqrt(d_k)
    
    # Apply mask (for causal attention)
    if mask is not None:
        scores = np.where(mask, scores, -1e9)
    
    # Softmax to get attention weights
    weights = softmax(scores, axis=-1)
    
    # Weighted sum of values
    output = weights @ V
    
    return output, weights

In [ ]:
# Example: Simple attention
seq_len, d_k = 4, 8
Q = np.random.randn(seq_len, d_k)
K = np.random.randn(seq_len, d_k)
V = np.random.randn(seq_len, d_k)

output, weights = scaled_dot_product_attention(Q, K, V)

print(f"Q shape: {Q.shape}")
print(f"K shape: {K.shape}")
print(f"V shape: {V.shape}")
print(f"Output shape: {output.shape}")
print(f"Weights shape: {weights.shape}")
print(f"\nAttention weights (rows sum to 1):")
print(weights.round(3))

In [ ]:
# Visualize attention weights
def visualize_attention(weights, title="Attention Weights"):
    plt.figure(figsize=(8, 6))
    plt.imshow(weights, cmap='Blues')
    plt.colorbar()
    plt.xlabel('Key position')
    plt.ylabel('Query position')
    plt.title(title)
    
    # Add values
    for i in range(weights.shape[0]):
        for j in range(weights.shape[1]):
            plt.text(j, i, f'{weights[i,j]:.2f}', ha='center', va='center')
    
    plt.show()

visualize_attention(weights)

## 5.2 Causal (Masked) Attention

In [ ]:
def create_causal_mask(seq_len):
    """Create a causal mask (lower triangular)."""
    return np.tril(np.ones((seq_len, seq_len), dtype=bool))

# Test causal attention
mask = create_causal_mask(seq_len)
output_causal, weights_causal = scaled_dot_product_attention(Q, K, V, mask)

print("Causal mask:")
print(mask.astype(int))
print("\nCausal attention weights:")
print(weights_causal.round(3))

In [ ]:
# Compare regular vs causal attention
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

im1 = axes[0].imshow(weights, cmap='Blues')
axes[0].set_title('Bidirectional Attention')
axes[0].set_xlabel('Key position')
axes[0].set_ylabel('Query position')
plt.colorbar(im1, ax=axes[0])

im2 = axes[1].imshow(weights_causal, cmap='Blues')
axes[1].set_title('Causal (Masked) Attention')
axes[1].set_xlabel('Key position')
axes[1].set_ylabel('Query position')
plt.colorbar(im2, ax=axes[1])

plt.tight_layout()
plt.show()

## 5.3 Multi-Head Attention

In [ ]:
class MultiHeadAttention:
    """Multi-head attention mechanism."""
    
    def __init__(self, d_model, num_heads):
        assert d_model % num_heads == 0
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        # Projection matrices
        scale = np.sqrt(2.0 / d_model)
        self.W_q = np.random.randn(d_model, d_model) * scale
        self.W_k = np.random.randn(d_model, d_model) * scale
        self.W_v = np.random.randn(d_model, d_model) * scale
        self.W_o = np.random.randn(d_model, d_model) * scale
    
    def split_heads(self, x):
        """Split into multiple heads: (batch, seq, d_model) -> (batch, heads, seq, d_k)"""
        batch_size = x.shape[0] if x.ndim == 3 else 1
        if x.ndim == 2:
            x = x[np.newaxis, ...]
        
        seq_len = x.shape[1]
        x = x.reshape(batch_size, seq_len, self.num_heads, self.d_k)
        return x.transpose(0, 2, 1, 3)  # (batch, heads, seq, d_k)
    
    def combine_heads(self, x):
        """Combine heads: (batch, heads, seq, d_k) -> (batch, seq, d_model)"""
        batch_size, _, seq_len, _ = x.shape
        x = x.transpose(0, 2, 1, 3)  # (batch, seq, heads, d_k)
        return x.reshape(batch_size, seq_len, self.d_model)
    
    def forward(self, x, mask=None):
        """Forward pass."""
        # Project to Q, K, V
        Q = x @ self.W_q
        K = x @ self.W_k
        V = x @ self.W_v
        
        # Split into heads
        Q = self.split_heads(Q)
        K = self.split_heads(K)
        V = self.split_heads(V)
        
        # Apply attention
        output, weights = scaled_dot_product_attention(Q, K, V, mask)
        
        # Combine heads
        output = self.combine_heads(output)
        
        # Final projection
        output = output @ self.W_o
        
        return output.squeeze(0), weights

In [ ]:
# Test multi-head attention
d_model, num_heads = 64, 8
mha = MultiHeadAttention(d_model, num_heads)

x = np.random.randn(10, d_model)  # 10 tokens, 64 dim
output, weights = mha.forward(x)

print(f"Input shape: {x.shape}")
print(f"Output shape: {output.shape}")
print(f"Weights shape: {weights.shape}")
print(f"  (batch, heads, seq_q, seq_k)")

In [ ]:
# Visualize attention patterns from different heads
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for i, ax in enumerate(axes.flat):
    im = ax.imshow(weights[0, i], cmap='Blues')
    ax.set_title(f'Head {i+1}')
    ax.set_xlabel('Key')
    ax.set_ylabel('Query')

plt.suptitle('Attention Patterns Across Heads', fontsize=14)
plt.tight_layout()
plt.show()

## 5.4 Positional Encoding

In [ ]:
def positional_encoding(max_len, d_model):
    """
    Sinusoidal positional encoding.
    
    PE(pos, 2i) = sin(pos / 10000^(2i/d_model))
    PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))
    """
    pe = np.zeros((max_len, d_model))
    position = np.arange(max_len)[:, np.newaxis]
    div_term = np.exp(np.arange(0, d_model, 2) * (-np.log(10000.0) / d_model))
    
    pe[:, 0::2] = np.sin(position * div_term)
    pe[:, 1::2] = np.cos(position * div_term)
    
    return pe

In [ ]:
# Visualize positional encoding
pe = positional_encoding(100, 64)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Heatmap
im = axes[0].imshow(pe, cmap='RdBu', aspect='auto')
axes[0].set_xlabel('Dimension')
axes[0].set_ylabel('Position')
axes[0].set_title('Positional Encoding Heatmap')
plt.colorbar(im, ax=axes[0])

# Individual dimensions
for i in [0, 1, 4, 5, 10, 11]:
    axes[1].plot(pe[:, i], label=f'dim {i}')
axes[1].set_xlabel('Position')
axes[1].set_ylabel('Value')
axes[1].set_title('Positional Encoding by Dimension')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5.5 Transformer Block

In [ ]:
class LayerNorm:
    """Layer normalization."""
    def __init__(self, d_model, eps=1e-6):
        self.gamma = np.ones(d_model)
        self.beta = np.zeros(d_model)
        self.eps = eps
    
    def forward(self, x):
        mean = x.mean(axis=-1, keepdims=True)
        std = x.std(axis=-1, keepdims=True)
        return self.gamma * (x - mean) / (std + self.eps) + self.beta

class FeedForward:
    """Position-wise feed-forward network."""
    def __init__(self, d_model, d_ff):
        scale = np.sqrt(2.0 / d_model)
        self.W1 = np.random.randn(d_model, d_ff) * scale
        self.b1 = np.zeros(d_ff)
        self.W2 = np.random.randn(d_ff, d_model) * scale
        self.b2 = np.zeros(d_model)
    
    def forward(self, x):
        # ReLU or GELU activation
        hidden = np.maximum(0, x @ self.W1 + self.b1)
        return hidden @ self.W2 + self.b2

class TransformerBlock:
    """A single Transformer block."""
    def __init__(self, d_model, num_heads, d_ff):
        self.mha = MultiHeadAttention(d_model, num_heads)
        self.ff = FeedForward(d_model, d_ff)
        self.norm1 = LayerNorm(d_model)
        self.norm2 = LayerNorm(d_model)
    
    def forward(self, x, mask=None):
        # Self-attention with residual
        attn_out, weights = self.mha.forward(self.norm1.forward(x), mask)
        x = x + attn_out
        
        # Feed-forward with residual
        ff_out = self.ff.forward(self.norm2.forward(x))
        x = x + ff_out
        
        return x, weights

In [ ]:
# Test Transformer block
d_model, num_heads, d_ff = 64, 8, 256
block = TransformerBlock(d_model, num_heads, d_ff)

x = np.random.randn(20, d_model)  # 20 tokens
output, weights = block.forward(x)

print(f"Input shape: {x.shape}")
print(f"Output shape: {output.shape}")
print(f"Input mean: {x.mean():.4f}, std: {x.std():.4f}")
print(f"Output mean: {output.mean():.4f}, std: {output.std():.4f}")

## Summary

In this notebook, we:
1. Implemented scaled dot-product attention
2. Created causal masks for autoregressive models
3. Built multi-head attention
4. Generated sinusoidal positional encodings
5. Assembled a complete Transformer block

**Next:** Module 6 covers Large Language Models.